# D.R.O.N.A. — ACT Gesture Policy Training (C3)

End-to-end walkthrough of training ACT (Action Chunking with Transformers) policies for the SO-100 arm gestures.

**Sections:**
1. Collect demonstration episodes from keyframe interpolation
2. Inspect episode quality (jerk, smoothness, trajectory shape)
3. Train ACT policy per gesture using LeRobot
4. Evaluate: compare ACT vs keyframe jerk scores
5. Save training curves

**Prerequisites:**
- `pip install lerobot torch` (lerobot >= 0.3)
- GPU recommended (GTX 1650 minimum)

**Approximate training time (GTX 1650):**
- Per gesture: ~20 minutes at 150 epochs
- All 6 gestures: ~2 hours

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from drona.utils.logging import setup_logging
setup_logging('INFO')

from drona.interaction.demonstration import (
    GESTURE_KEYFRAMES, interpolate_keyframes, REST_POSE, DOF, JOINT_NAMES
)
from drona.interaction.act_policy import PolicyRouter, KeyframePolicy

print('Available gestures:', list(GESTURE_KEYFRAMES.keys()))
print('DOF:', DOF)
print('Joint names:', JOINT_NAMES)

## 1. Inspect Keyframe Trajectories

In [ ]:
def compute_jerk(traj_q, dt=0.05):
    """Compute per-joint jerk from a joint-angle trajectory."""
    q = np.array(traj_q)  # (T, DOF)
    vel  = np.diff(q, axis=0) / dt
    acc  = np.diff(vel, axis=0) / dt
    jerk = np.diff(acc, axis=0) / dt
    return jerk

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

jerk_summary = {}
for idx, (gesture, kfs) in enumerate(GESTURE_KEYFRAMES.items()):
    traj = interpolate_keyframes(kfs, dt=0.05)
    qs = np.array([q for q, _ in traj])
    
    jerk = compute_jerk(qs)
    peak_jerk = float(np.abs(jerk).max())
    rms_jerk = float(np.sqrt(np.mean(jerk**2)))
    jerk_summary[gesture] = {'peak': peak_jerk, 'rms': rms_jerk, 'steps': len(qs)}
    
    ax = axes[idx]
    t = np.linspace(0, len(qs) * 0.05, len(qs))
    for j in range(DOF):
        ax.plot(t, qs[:, j], label=JOINT_NAMES[j] if len(JOINT_NAMES) > j else f'J{j}', alpha=0.8)
    ax.set_title(f'{gesture} | peak jerk={peak_jerk:.1f} rad/s³')
    ax.set_xlabel('time (s)')
    ax.set_ylabel('angle (rad)')
    if idx == 0:
        ax.legend(fontsize=7, loc='upper right')

plt.suptitle('Keyframe Gesture Trajectories', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/evaluation/gesture_trajectories_keyframe.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nKeyframe jerk summary:')
for gesture, s in jerk_summary.items():
    print(f"  {gesture:10s}: peak={s['peak']:6.2f} rad/s³  rms={s['rms']:5.2f} rad/s³  steps={s['steps']}")

## 2. Collect Training Episodes

In [ ]:
from drona.interaction.mujoco_env import StubEnv

EPISODES_PER_GESTURE = 50
CHUNK_SIZE = 10
DT = 0.05
NOISE_STD = 0.01  # rad — adds slight variation to each episode

def collect_episodes(gesture: str, n_episodes: int = EPISODES_PER_GESTURE):
    """Collect demonstration episodes by replaying keyframes with small noise."""
    kfs = GESTURE_KEYFRAMES[gesture]
    episodes = []
    
    for ep in range(n_episodes):
        traj = interpolate_keyframes(kfs, dt=DT)
        # Add small Gaussian noise to simulate hardware variation
        obs_list, act_list = [], []
        env = StubEnv(dt=DT)
        obs = env.reset()
        
        for q_target, _ in traj:
            noise = np.random.normal(0, NOISE_STD, size=q_target.shape)
            obs_list.append(obs.copy())
            act_list.append((q_target + noise).astype(np.float32))
            obs, _ = env.step(q_target)
        
        env.close()
        # Build (obs, action_chunk) pairs
        for t in range(len(obs_list) - CHUNK_SIZE):
            episodes.append({
                'obs': obs_list[t],
                'actions': np.array(act_list[t:t + CHUNK_SIZE]),
            })
    
    return episodes

# Demo: collect for 'greet' only
print('Collecting greet episodes...')
greet_eps = collect_episodes('greet', n_episodes=5)
print(f'Collected {len(greet_eps)} (obs, action_chunk) pairs')
print(f'obs shape:     {greet_eps[0]["obs"].shape}')
print(f'actions shape: {greet_eps[0]["actions"].shape}  ({CHUNK_SIZE} steps × {DOF} joints)')

## 3. Train ACT Policy

In [ ]:
try:
    import lerobot
    import torch
    HAS_LEROBOT = True
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'LeRobot {lerobot.__version__} ready. Device: {device}')
    if device == 'cuda':
        print(f'GPU: {torch.cuda.get_device_name(0)}')
        print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
except ImportError:
    HAS_LEROBOT = False
    print('LeRobot not installed. Run: pip install lerobot')
    print('Skipping training cells — jerk comparison uses keyframe data only.')

In [ ]:
%%time
# Train for a SINGLE gesture (greet) as demonstration.
# For all gestures: python scripts/train_act.py

if HAS_LEROBOT:
    from scripts.train_act import _build_torch_dataset, _make_act_policy, _train_gesture
    from pathlib import Path
    
    GESTURE = 'greet'
    N_EPISODES = 50
    EPOCHS = 30  # Quick demo — use 150 for full training
    CHECKPOINT_DIR = Path('../data/checkpoints')
    
    print(f'Training ACT for "{GESTURE}" — {EPOCHS} epochs...')
    
    episodes = collect_episodes(GESTURE, n_episodes=N_EPISODES)
    dataset = _build_torch_dataset(episodes, chunk_size=CHUNK_SIZE)
    policy = _make_act_policy(dof=DOF, chunk_size=CHUNK_SIZE)
    
    loss_history = _train_gesture(
        gesture=GESTURE,
        policy=policy,
        dataset=dataset,
        epochs=EPOCHS,
        batch_size=32,
        lr=1e-4,
        checkpoint_dir=CHECKPOINT_DIR,
        device=device,
    )
    
    # Plot training curve
    plt.figure(figsize=(8, 4))
    plt.plot(loss_history, color='steelblue')
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.title(f'ACT Training Loss — {GESTURE} ({EPOCHS} epochs)')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'../data/evaluation/act_training_{GESTURE}.png', dpi=150)
    plt.show()
    print(f'Final loss: {loss_history[-1]:.6f}')
    print(f'Checkpoint saved to {CHECKPOINT_DIR}/{GESTURE}/')
else:
    print('Skipped (LeRobot not installed).')

## 4. Jerk Comparison: ACT vs Keyframe

In [ ]:
from drona.interaction.mujoco_env import StubEnv
import json, os

def rollout_policy(policy, n_steps=60, dt=0.05):
    """Roll out a policy and return joint trajectory."""
    env = StubEnv(dt=dt)
    obs = env.reset()
    policy.reset()
    qs = []
    for _ in range(n_steps):
        action = policy.select_action({'observation.state': obs})
        obs, _ = env.step(action)
        qs.append(obs.copy())
        if getattr(policy, 'is_complete', False):
            break
    env.close()
    return np.array(qs)

router = PolicyRouter(checkpoint_base_dir='../data/checkpoints', device='cpu')
comparison = {}

for gesture in GESTURE_KEYFRAMES:
    policy = router.get_policy(gesture)
    policy_type = type(policy).__name__
    
    qs = rollout_policy(policy)
    if len(qs) >= 4:
        jerk = compute_jerk(qs)
        peak = float(np.abs(jerk).max())
        rms = float(np.sqrt(np.mean(jerk**2)))
    else:
        peak = rms = 0.0
    
    comparison[gesture] = {
        'policy_type': policy_type,
        'peak_jerk': peak,
        'rms_jerk': rms,
        'n_steps': len(qs),
    }
    print(f'{gesture:10s} [{policy_type:20s}]: peak={peak:6.2f} rad/s³  rms={rms:5.2f}')

# Plot comparison
gestures = list(comparison.keys())
peak_jerks = [comparison[g]['peak_jerk'] for g in gestures]
types = [comparison[g]['policy_type'] for g in gestures]
colors = ['#4CAF50' if 'ACT' in t else '#FF9800' for t in types]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(gestures, peak_jerks, color=colors)
ax.axhline(30, color='red', linestyle='--', linewidth=1.5, label='C3 target (30 rad/s³)')
ax.axhline(50, color='orange', linestyle=':', linewidth=1.5, label='Acceptable (50 rad/s³)')
ax.set_ylabel('Peak Jerk (rad/s³)')
ax.set_title('Gesture Smoothness — Peak Jerk by Policy Type')
ax.legend()
from matplotlib.patches import Patch
legend_patches = [Patch(color='#4CAF50', label='ACT policy'), Patch(color='#FF9800', label='Keyframe policy')]
ax.legend(handles=legend_patches + [plt.Line2D([0], [0], color='red', linestyle='--', label='C3 target (30 rad/s³)'),
                                      plt.Line2D([0], [0], color='orange', linestyle=':', label='Acceptable (50 rad/s³)')])
for bar, v in zip(bars, peak_jerks):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, f'{v:.1f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('../data/evaluation/c3_jerk_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# Save JSON results
os.makedirs('../data/evaluation', exist_ok=True)
with open('../data/evaluation/c3_gesture_smoothness.json', 'w') as f:
    json.dump(comparison, f, indent=2)
print('\nResults saved to data/evaluation/c3_gesture_smoothness.json')

# C3 verdict
act_gestures = {g: v for g, v in comparison.items() if 'ACT' in v['policy_type']}
if act_gestures:
    max_peak = max(v['peak_jerk'] for v in act_gestures.values())
    print(f'C3 result: max ACT peak jerk = {max_peak:.2f} rad/s³')
    print(f'C3 status: {"✅ PASS" if max_peak <= 50 else "❌ FAIL"} (target ≤ 30, acceptable ≤ 50)')
else:
    print('No ACT policies found. Train with: python scripts/train_act.py')